In [1]:
from yourbench.pipeline.single_shot_question_generation import _load_stage_config
import random
from typing import Any
from dataclasses import field, dataclass

from loguru import logger

from datasets import Dataset
from yourbench.utils.prompts import (
    QUESTION_GENERATION_USER_PROMPT,
    QUESTION_GENERATION_SYSTEM_PROMPT,
    QUESTION_GENERATION_SYSTEM_PROMPT_MULTI,
)
from yourbench.utils.dataset_engine import (
    custom_load_dataset,
    custom_save_dataset,
)

# Import the unified parsing function
from yourbench.utils.parsing_engine import shuffle_mcq, parse_qa_pairs_from_response
from yourbench.utils.inference_engine import InferenceCall, run_inference
from yourbench.utils.loading_engine import load_config



In [2]:
config=load_config("./my_example.yaml")

2025-05-28 11:05:24.860 | DEBUG    | yourbench.utils.loading_engine:load:42 - Read configuration from my_example.yaml
2025-05-28 11:05:24.870 | DEBUG    | yourbench.utils.loading_engine:load:46 - Configuration loaded successfully from my_example.yaml


In [3]:
config.keys()

dict_keys(['hf_configuration', 'model_list', 'model_roles', 'pipeline'])

In [39]:
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk, concatenate_datasets
stage_config = _load_stage_config(config)
chunked_dataset = load_from_disk(dataset_path="./example/data/local_saved/chunked")
summarized_dataset = load_from_disk(dataset_path="./example/data/local_saved/summarized")

In [5]:
local_dataset_dir = config["hf_configuration"].get("local_dataset_dir")
local_dataset_dir

In [6]:
'yes'if local_dataset_dir else 'no'

'no'

In [40]:
dataset=concatenate_datasets([chunked_dataset, summarized_dataset])

In [41]:
len(dataset)

4

In [43]:
n=len(dataset)

for row_index, row in zip(range(n), dataset):
    print("---"*20)
    print("row_index:", row_index, "\n", row.keys())
    print("chunks =\n", row["chunks"])
    print("--.---"*15)
    print("document_summary =\n", srow["document_summary"])
    

------------------------------------------------------------
row_index: 0 
 dict_keys(['document_id', 'document_text', 'document_filename', 'document_metadata', 'chunks', 'multihop_chunks', 'chunk_info_metrics', 'chunking_model', 'raw_chunk_summaries', 'chunk_summaries', 'raw_document_summary', 'document_summary', 'summarization_model'])
chunks =
 [{'chunk_id': '96a62262-f99a-4536-9cd0-edb986703892_0', 'chunk_text': 'Hurricanes: Interesting Facts and F. A. Q. \uf0a7  The word hurricane comes from the Taino Native American word, hurucane, meaning  evil spirit of the wind. \uf0a7  The first time anyone flew into a hurricane happened in 1943 in the middle of World  War II. \uf0a7  A tropical storm is classified as a hurricane once winds goes up to 74 miles per hour or  higher. \uf0a7  Hurricanes are the only weather disasters that have been given their own names. \uf0a7  All hurricanes begin life in a warm moist atmosphere over tropical ocean waters. \uf0a7  A'}, {'chunk_id': '96a62262-f9